# Step 1 & 2: Experiment with managed MLflow, then scale with SageMaker AI jobs

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.3.2</strong> and the <strong>SageMaker Python SDK version 3.20.0</strong>.
</div>

This notebook walks the full path from *idea* to a *scaled, tracked, registered, deployed model* in one pass, so nothing is computed twice.

**From idea to production in four steps:**
||||
|---|---|---|
|1. |Experiment in a notebook |**<<<< YOU ARE HERE (Part 1)**|
|2. |Scale with SageMaker AI processing and training jobs |**<<<< YOU ARE HERE (Part 2)**|
|3. |Operationalize with an ML pipeline and model registry ||
|4. |Add model deployment and monitoring ||

## What you do here

**Part 1 — Prototype in the notebook.** Explore the bank-marketing dataset, engineer features, train XGBoost locally, and track every run in [SageMaker managed MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html). No `sagemaker` job APIs — just plain Python. This is the fast, cheap iteration loop. Nothing from this part is registered: the prototype stays an experiment.

**Part 2 — Move the same logic onto SageMaker AI managed infrastructure.** SageMaker uses [Docker containers](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html) to process data, train, and deploy models. Containers package code, runtime, system libraries, and settings into one standardized unit, so a job behaves identically wherever it runs. Here you use the custom XGBoost image (BYOC) you built in notebook 00 for processing, training, *and* inference. The model produced by the managed training job is the one that gets registered, approved, and deployed.

You will use, each exactly once:

| Capability | Where |
|---|---|
| `FrameworkProcessor` on managed compute | Data processing at scale |
| `@remote` decorator | Simplest lift of a Python function into a training job |
| `ModelTrainer` in `LOCAL_CONTAINER` mode | Validate the script/container contract in Docker before paying for instances |
| `ModelTrainer` in `SAGEMAKER_TRAINING_JOB` mode | Training at scale |
| `@remote` on a shared step function | SHAP explainability + fairness report, run as a managed job |
| Inference specification logging + `mlflow.register_model()` | Make the model deployable, then register it — auto-sync creates a **deployable** SageMaker Model Package |
| Direct import of the logged `inference.py` | Smoke-test the real serving code in this kernel before approving the package |
| `Model` / `EndpointConfig` / `Endpoint` (SDK v3) | Deploy the **approved Model Package version** to a real-time endpoint |

<div class="alert alert-info">
<b>No duplicated effort:</b> the preprocessing logic is prototyped once in Part 1 and then executed once as a managed job in Part 2. The datasets written to S3 by that job are the canonical inputs for notebooks 03 and 04. There is exactly one registration and one real endpoint, and the endpoint serves the registered package.
</div>

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/aim-408-end-to-end-modelops-on-sagemaker-ai
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the top

</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

### Check SageMaker version installed

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

### Import packages
One import cell for the whole notebook. The two `Mode` enums (training and serving) are aliased, because the SDK gives them the same class name.

In [ ]:
import os
import time
import json
import importlib.util
import tempfile
import warnings
from io import StringIO
from importlib.metadata import version

import boto3
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import mlflow
import sagemaker_mlflow
from mlflow import MlflowClient
from IPython.display import HTML, Javascript
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score

from sagemaker.core.resources import (
    MlflowApp,
    Model,
    ModelPackage,
    EndpointConfig,
    Endpoint,
)
from sagemaker.core.s3 import S3Downloader
from sagemaker.core.s3.client import s3_path_join
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.utils.utils import configure_logging
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output,
    OutputDataConfig,
    ContainerDefinition,
    ProductionVariant,
)
from sagemaker.core import FrameworkProcessor
from sagemaker.core.remote_function.client import remote
from sagemaker.core.remote_function.custom_file_filter import CustomFileFilter
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, SourceCode, Compute
from sagemaker.train.model_trainer import Mode as TrainingMode

# The explainability step is shared with notebook 03, where the same function
# is wrapped with @step instead of @remote.
from pipeline_steps.explain import explain

warnings.filterwarnings("ignore")
configure_logging("INFO")

Load the stored variables from previous notebooks.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In addition to the variables stored previously, let's set and store the name of the column containing the target variable in our dataset.

In [ ]:
target_col = "y"
%store target_col

## Set up experiment tracking with managed MLflow
You can [manage machine learning experiments using Amazon SageMaker with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) through serverless MLflow apps managed by SageMaker.

[MLflow tracking](https://mlflow.org/docs/latest/tracking.html) automatically records the inputs, parameters, configurations, and models of your iterations as `experiments` and `runs`. See [MLflow concepts](https://mlflow.org/docs/latest/tracking.html#concepts) for how runs and experiments are organized.

Notebook 00 created (or reused) the MLflow app and stored its ARN in `mlflow_arn`. The cell below waits until the app is ready — this can take 2-3 minutes on a first run.

In [ ]:
mlflow_app = MlflowApp.get(arn=mlflow_arn)

max_wait_time = 300  # 5 minutes
start_time = time.time()

while mlflow_app.status in ["Creating", "Updating"]:
    if time.time() - start_time > max_wait_time:
        raise TimeoutError(f"MLflow app '{mlflow_name}' still {mlflow_app.status} after 5 minutes")
    print(f"MLflow app '{mlflow_name}' is {mlflow_app.status}... waiting 30 seconds")
    time.sleep(30)
    mlflow_app.refresh()

if mlflow_app.status not in ["Created", "Updated"]:
    raise RuntimeError(f"MLflow app '{mlflow_name}' is not usable (status: {mlflow_app.status})")

print(f"MLflow app '{mlflow_name}' is ready (status: {mlflow_app.status})")

Point MLflow at the app and set the names for the experiment and the registered model.

These are **shared names**: one experiment and one registered model for the whole workshop, suffixed with a short date stamp (`-DDMMYY`). They are created here on first use and persisted with `%store`, so every later notebook restores and reuses the same names — `set_experiment()` returns the existing experiment, and each `register_model()` call adds a new **version** of the same registered model rather than creating a new model.

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)
print(f"Using MLflow server: {mlflow_arn}")

date_suffix = time.strftime("%d%m%y", time.gmtime())
experiment_name = f"from-idea-to-prod-experiment-{date_suffix}"
registered_model_name = f"from-idea-to-prod-model-{date_suffix}"

%store experiment_name
%store registered_model_name

experiment = mlflow.set_experiment(experiment_name=experiment_name)

print(f"Experiment: {experiment_name}")
print(f"Registered model: {registered_model_name}")

---
# Part 1 — Prototype in the notebook

## Load data

The dataset is already available locally from notebook 00.

In [ ]:
file_name = "bank-additional-full.csv"
input_path = "./data/bank-additional" 
output_path = "./data"

## EDA
Let's do some exploratory data analysis on this dataset.

In [ ]:
df_data = pd.read_csv(os.path.join(input_path, file_name), sep=";")

pd.set_option("display.max_columns", 500)  # View all of the columns
df_data  # show first 5 and last 5 rows of the dataframe

Take a look at the column metadata to understand how many null values and which data types you're dealing with.

In [ ]:
df_data.info()

Looking at the column statistics can be a good, quick sanity check. For example, the minimum value for age is 17, the maximum value is 98, and the mean is 40, which all seems reasonable.

In [ ]:
df_data.describe()

Next, take a look at the distribution of the target column. This shows us that the dataset is unbalanced, with more "no" than "yes" results for marketing calls.

In [ ]:
df_data[target_col].value_counts().plot.bar()
plt.show()

Double-check if you're dealing with any missing values. In this case, the dataset is very clean.

In [ ]:
df_data.isna().sum()

Now let's take a look at the distribution for some of the remaining columns. From the results we can see that most customers are educated to a high degree, most are married, and we have some with an unknown job.

In [ ]:
cat_columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

fig, axs = plt.subplots(3, 3, sharex=False, sharey=False, figsize=(20, 15))

counter = 0
for cat_column in cat_columns:
    value_counts = df_data[cat_column].value_counts()
    
    trace_x = counter // 3
    trace_y = counter % 3
    x_pos = np.arange(0, len(value_counts))
    
    axs[trace_x, trace_y].bar(x_pos, value_counts.values, tick_label = value_counts.index)
    
    axs[trace_x, trace_y].set_title(cat_column)
    
    for tick in axs[trace_x, trace_y].get_xticklabels():
        tick.set_rotation(90)
    
    counter += 1

plt.show()

Do a similar distribution check for numerical columns. Here we can see that the duration of the call is almost always very short, and that the majority of customers were not contacted recently for a previous campaign (`pdays`).

In [ ]:
num_columns = ['duration', 'campaign', 'pdays', 'previous']

fig, axs = plt.subplots(2, 2, sharex=False, sharey=False, figsize=(20, 15))

counter = 0
for num_column in num_columns:
    
    trace_x = counter // 2
    trace_y = counter % 2
    
    axs[trace_x, trace_y].hist(df_data[num_column])
    
    axs[trace_x, trace_y].set_title(num_column)
    
    counter += 1

plt.show()

The following cell shows how to compare specific columns with the target variable, in this case marital status. 

In [ ]:
j_df = pd.DataFrame()

j_df['yes'] = df_data[df_data[target_col] == 'yes']['marital'].value_counts()
j_df['no'] = df_data[df_data[target_col] == 'no']['marital'].value_counts()

j_df.plot.bar(title = 'Marital status and deposit')

## Feature engineering in the notebook
You prototype the transformations here. In Part 2 the exact same logic moves into `processing/preprocessing.py` and runs as a SageMaker processing job — it is written once here and executed once as a job later.

The processing implements:
1. Create a new column called `no_previous_contact`. Set value to `1` when `pdays` is `999` and `0` otherwise
1. Generate a new column to show whether the customer is working based on the `job` column
1. Remove the economic features from the dataset, as they would need to be forecasted with high precision to be usable as features at inference time
1. Remove `duration`, as it is not known before a call is performed
1. Convert categorical variables to numeric using **one hot encoding**
1. Move the target column `y` to the front

In the real world you implement additional processing, data quality handling, and feature engineering, and you go through multiple "try & fail" iterations.

In [ ]:
# Indicator variable to capture when pdays takes a value of 999
df_data["no_previous_contact"] = np.where(df_data["pdays"] == 999, 1, 0)

# Indicator for individuals not actively employed
df_data["not_working"] = np.where(
    np.isin(df_data["job"], ["student", "retired", "unemployed"]), 1, 0
)

# remove unnecessary data
df_model_data = df_data.drop(
    ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
    axis=1,
)


bins = [18, 30, 40, 50, 60, 70, 90]
labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70-plus']

df_model_data['age_range'] = pd.cut(df_model_data.age, bins, labels=labels, include_lowest=True)
df_model_data = pd.concat([df_model_data, pd.get_dummies(df_model_data['age_range'], prefix='age', dtype=int)], axis=1)
df_model_data.drop('age', axis=1, inplace=True)
df_model_data.drop('age_range', axis=1, inplace=True)

scaled_features = ['pdays', 'previous', 'campaign']
df_model_data[scaled_features] = MinMaxScaler().fit_transform(df_model_data[scaled_features])

df_model_data = pd.get_dummies(df_model_data, dtype=int)  # Convert categorical variables to sets of indicators

# Replace "y_no" and "y_yes" with a single label column, and bring it to the front:
df_model_data = pd.concat(
    [
        df_model_data["y_yes"].rename(target_col),
        df_model_data.drop(["y_no", "y_yes"], axis=1),
    ],
    axis=1,
)

After running the preprocessing code, take another look at the dataset itself and some of the column statistics for a quick sanity check.

In [ ]:
df_model_data

In [ ]:
df_model_data.describe()

## Split data

[SageMaker's XGBoost algorithm](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html#InputOutput-XGBoost), which you use in Part 2 of this notebook, expects data in the libSVM or CSV formats, with:

- The target variable in the first column, and
- No header row 

Therefore, the cells below first shuffle and split the dataset into train, validation, and test sets before saving each dataset as a CSV file with the correct formatting.

In [ ]:
df_shuffled = df_model_data.sample(frac=1, random_state=1729)
# Split data into 70% training, 20% validation, 10% test
train_idx = int(0.7 * len(df_shuffled))
val_idx = int(0.9 * len(df_shuffled))

train_data = df_shuffled.iloc[:train_idx]
validation_data = df_shuffled.iloc[train_idx:val_idx]
test_data = df_shuffled.iloc[val_idx:]

print(f"Data split > train:{train_data.shape} | validation:{validation_data.shape} | test:{test_data.shape}")

In [ ]:
train_data.to_csv(os.path.join(output_path, "train.csv"), index=False, header=False)
validation_data.to_csv(os.path.join(output_path, "validation.csv"), index=False, header=False)
test_data.to_csv(os.path.join(output_path, "test.csv"), index=False, header=False)

### Track the feature engineering in an MLflow run
Log your data processing, not only your training, so that preprocessing and training information is grouped in the same experiment.

Create a run with [`mlflow.start_run()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.start_run), then use [`log_params()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_params) and [`log_artifact()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_artifact) to record information into it. `mlflow.log_artifact()` uploads a local file to the MLflow artifact store under the S3 URI you specified when the MLflow app was created.

Here you log the split ratios and the **raw input dataset**. The produced train/validation/test splits are logged later by the processing job in Part 2, so the same CSVs are not uploaded twice.

In [ ]:
with mlflow.start_run(
    run_name=name_from_base("feature-engineering", short=True),
    description="feature engineering prototyped in the notebook (Part 1)"
) as run:
    mlflow.set_tags({"mlflow.user": user_profile_name})
    mlflow.log_params(
        {
            "train": 0.7,
            "validate": 0.2,
            "test": 0.1
        }
    )
    # Log the raw input dataset. The train/validation/test splits produced from it
    # are logged by the SageMaker processing job in Part 2 (same data, logged once).
    mlflow.log_artifact(local_path=os.path.join(input_path, file_name))

## Model training and validation

In this notebook, you will use the [open-source version of XGBoost](https://xgboost.ai/about) since we are demonstrating full local processing and training inside the notebook.

First, split the features from the label in the training dataset.

In [ ]:
train_features = train_data.drop(target_col, axis=1)
train_label = pd.DataFrame(train_data[target_col])

Create the data structure which the XGBoost algorithm accepts as input.

In [ ]:
dtrain = xgb.DMatrix(train_features, label=train_label)

Set your hyperparameters and other variables.

In [ ]:
hyperparams = {
                "max_depth": 5,
                "eta": 0.5,
                "alpha": 2.5,
                "objective": "binary:logistic",
                "subsample" : 0.8,
                "colsample_bytree" : 0.8,
                "min_child_weight" : 3
              }

num_boost_round = 150
nfold = 3
early_stopping_rounds = 10

First, you can train the model on `nfold` number of folds of the training dataset and run a cross-validation. This helps us understand how well the model generalizes with the current hyperparameters.

In [ ]:
cv_results = xgb.cv(
    params=hyperparams,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    nfold=nfold,
    early_stopping_rounds=early_stopping_rounds,
    metrics=["auc"],
    seed=10,
)

Fetch and print the metrics to understand how well the model performs.

In [ ]:
metrics_data = {
    "binary_classification_metrics": {
        "validation:auc": {
            "value": cv_results.iloc[-1]["test-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["test-auc-std"]
        },
        "train:auc": {
            "value": cv_results.iloc[-1]["train-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["train-auc-std"]
        },
    }
}

print(f"Cross-validated train-auc:{cv_results.iloc[-1]['train-auc-mean']:.2f}")
print(f"Cross-validated validation-auc:{cv_results.iloc[-1]['test-auc-mean']:.2f}")
display(metrics_data)

cv_results

Next, you can retrain the model on the full training dataset instead of splitting the training dataset across a number of folds.

### Train a model
Use [MLflow model flavor](https://mlflow.org/docs/latest/python_api/index.html#python-api) and [logging functions](https://mlflow.org/docs/latest/tracking/tracking-api.html#tracking-logging-functions) to log parameters, model, model metrics, and various metadata in your experiment runs.

You will use the test dataset for early stopping. So first split the feature and target label and create the data structure expected by the algorithm.

In [ ]:
test_features = test_data.drop(target_col, axis=1)
test_label = pd.DataFrame(test_data[target_col])
dtest = xgb.DMatrix(test_features, label=test_label)

The cell below shows a simple training loop which trains the model for different `max_depth` values and logs each run in MLflow. This will enable you to easily compare the results later.

In [ ]:
max_metric = 0.0
best_model_run_id = 0

with mlflow.start_run(
    run_name=name_from_base("training-", short=True),
    description=f"Fit estimator with different max_depth"
) as parent_run:
    mlflow.set_tags({'mlflow.user':user_profile_name})
    
    # Train the model for different max_depth values
    for index, depth in enumerate([2, 5, 10, 15, 20]):
        hyperparams["max_depth"] = depth
        print(f"Fit estimator with max_depth={depth}")
    
        with mlflow.start_run(
            run_name=f"max_depth={depth}",
            description=f"Fit estimator with max_depth={depth}",
            nested=True
        ) as child_run:
            mlflow.set_tags({'mlflow.user':user_profile_name})
            
            mlflow.xgboost.autolog(silent=True)
            
            # Train the model
            model = xgb.train(
                params=hyperparams, 
                dtrain=dtrain, 
                evals = [(dtrain,'train'), (dtest,'eval')], 
                num_boost_round=num_boost_round, 
                early_stopping_rounds=early_stopping_rounds, 
                verbose_eval = 0
            )
    
            # Calculate metrics
            test_auc = roc_auc_score(test_label, model.predict(dtest))
            train_auc = roc_auc_score(train_label, model.predict(dtrain))
            
            # Log hyperparameters and metrics to the run
            mlflow.log_params(hyperparams)
            mlflow.log_metrics({"test_auc":test_auc, "train_auc":train_auc}, step=index)
    
            if test_auc > max_metric:
                best_model_run_id = child_run.info.run_id
                max_metric = test_auc
    
            print(f"Test AUC: {test_auc:.4f} | Train AUC: {train_auc:.4f}")

### The prototype is not registered — and that is deliberate

MLflow can act as a [model registry](https://mlflow.org/docs/latest/model-registry.html), and because the MLflow app runs with `AutoModelRegistrationEnabled`, every `mlflow.register_model()` call also creates a Model Package in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html). That makes registration a **governance action**, not a bookkeeping one: whatever lands in the registry is a candidate for approval and deployment.

A model trained in a notebook kernel does not qualify — it is not reproducible, has no training-job lineage, and carries no inference specification, so the synced Model Package would not be deployable. So Part 1 stops at tracked experiments. Part 2 registers exactly one model: the one produced by the managed training job, with an inference specification attached so the Model Package arrives ready to deploy.

For now, just note which local run won the sweep.

In [ ]:
print(f"Best local run: {best_model_run_id} (test AUC {max_metric:.4f})")
print("Tracked, not registered — Part 2 registers the model trained by the SageMaker training job.")

### Compare the local runs in the MLflow UI
Open the MLflow UI on the experiment you just populated and compare the `max_depth` runs in **Chart view**.

In [ ]:
sm_client = boto3.Session().client("sagemaker")

experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# get the presigned url to open the MLflow UI
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

compare_run_link = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/runs?workspace=default&compareRunsMode=CHART"

In [ ]:
# first open the MLflow UI - you can close a new opened window
display(Javascript('window.open("{}");'.format(presigned_url)))

In [ ]:
display(Javascript('window.open("{}");'.format(compare_run_link)))

## Explore experiments with the MLflow UI
As a starting point, you can access your MLflow app in the Studio UI in the **MLflow** widget > **Open MLflow** button. 

![](img/mlflow-apps-studio.png)

The MLflow UI opens in a new browser window. Here you can select your workspace (select *default*) and experiment (select experiment name we have generated) to view the runs within it.

![](img/experiment-mlflow.png)

You can switch to **Chart view** to compare different runs. The eye icons enable you to choose which runs to show or hide.

![](img/comparing-runs.png)

Change to the **Models** to see the registered models and all versions.

![](img/logged-models.png)

---
# Part 2 — Scale with SageMaker AI jobs

Everything above ran in this notebook's kernel: fine for prototyping on a subset, but it does not scale, is not reproducible, and cannot be scheduled. Next you move the identical logic into SageMaker managed jobs.

## Process data with SageMaker processing jobs
Use [SageMaker Processing](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html) by simply providing a Python script and choosing a [SageMaker SDK processor](https://sagemaker.readthedocs.io/en/stable/api/sagemaker_core.html#module-sagemaker.core.processing) class.

You can upload the input data to S3 and specify an S3 location for output data. SageMaker Processing automatically loads the input data from S3 and uploads transformed data back to S3 when the job is complete. The processing container image can either be an Amazon SageMaker built-in image or a custom image that you provide. The underlying infrastructure for a processing job is fully managed by Amazon SageMaker. Cluster resources are provisioned for the duration of your job, and cleaned up when a job completes.

![](img/sagemaker-processing.png)

### Why there is no separate "local processing" run
[Local mode](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local.html) lets you run a SageMaker job in a container on this JupyterLab space, which is useful for debugging a script before submitting it to managed compute. You already validated this preprocessing logic in Part 1 by running it directly in the notebook, and a local-mode processing job would write the *same* outputs to the *same* S3 prefixes as the managed job below — so it is skipped here to avoid doing the work twice.

Local mode is still demonstrated once in this notebook: see **Train with `ModelTrainer` in local container mode** further down, where it validates the training script against the container contract.

Note that while local mode works with SageMaker built-in containers and with your own container (BYOC), some built-in algorithm containers such as XGBoost do not support it.

Let's create a folder to store our processing code and requirements. The BYOC image already includes mlflow and the MLflow SageMaker plugin, so the `requirements.txt` here is effectively a no-op for this image — we keep it to demonstrate the mechanism `FrameworkProcessor` uses to install per-job dependencies, and so the same code still works if you switch back to a SageMaker built-in image that doesn't bundle them.

In [ ]:
local_processing_dir = "./processing/"
os.makedirs(local_processing_dir, exist_ok=True)
with open(os.path.join(local_processing_dir, "requirements.txt"), "w") as file_handle:
    file_handle.write("mlflow>=3,<4\nsagemaker-mlflow>=0.5.0,<1\n")

#### Create the processing script
Move the feature engineering code from Part 1 into a `.py` file and add MLflow tracking. Two things are added on top of the notebook version:

- the test set is written as separate `test_x.csv` / `test_y.csv` files, which is what the endpoint test and the pipeline in notebook 03 expect;
- the full feature set without the label is written as `baseline.csv`, the data baseline used by model monitoring in notebook 04.

In [ ]:
%%writefile processing/preprocessing.py
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import pandas as pd
import numpy as np
import argparse
import os

from time import gmtime, strftime, sleep
import traceback

user_profile_name = os.getenv('USER')
print(os.environ)
def _parse_args():
    
    parser = argparse.ArgumentParser()
    # Data, model, and output directories
    # model_dir is always passed in from SageMaker. By default this is a S3 path under the default bucket.
    parser.add_argument('--filepath', type=str, default='/opt/ml/processing/input/')
    parser.add_argument('--filename', type=str, default='bank-additional-full.csv')
    parser.add_argument('--outputpath', type=str, default='/opt/ml/processing/output/')
    
    return parser.parse_known_args()

def process_data(df_data):
    # Indicator variable to capture when pdays takes a value of 999
    df_data["no_previous_contact"] = np.where(df_data["pdays"] == 999, 1, 0)

    # Indicator for individuals not actively employed
    df_data["not_working"] = np.where(
        np.isin(df_data["job"], ["student", "retired", "unemployed"]), 1, 0
    )

    # remove unnecessary data
    df_model_data = df_data.drop(
        ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
        axis=1,
    )

    bins = [18, 30, 40, 50, 60, 70, 90]
    labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70-plus']

    df_model_data['age_range'] = pd.cut(df_model_data.age, bins, labels=labels, include_lowest=True)
    df_model_data = pd.concat([df_model_data, pd.get_dummies(df_model_data['age_range'], prefix='age', dtype=int)], axis=1)
    df_model_data.drop('age', axis=1, inplace=True)
    df_model_data.drop('age_range', axis=1, inplace=True)

    scaled_features = ['pdays', 'previous', 'campaign']
    df_model_data[scaled_features] = MinMaxScaler().fit_transform(df_model_data[scaled_features])

    df_model_data = pd.get_dummies(df_model_data, dtype=int)  # Convert categorical variables to sets of indicators

    # Replace "y_no" and "y_yes" with a single label column, and bring it to the front:
    df_model_data = pd.concat(
        [
            df_model_data["y_yes"].rename(target_col),
            df_model_data.drop(["y_no", "y_yes"], axis=1),
        ],
        axis=1,
    )
    
    return df_model_data

if __name__=="__main__":
    # Process arguments
    args, _ = _parse_args()
    
    target_col = "y"
    
    import mlflow
    
    # Set the Tracking Server URI using the ARN of the Tracking Server you created
    mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_ARN'])
    
    # Enable autologging in MLflow
    mlflow.autolog()

    # Use the active run_id to log 
    with mlflow.start_run(run_id=os.environ['MLFLOW_RUN_ID']) as run:
        # process data
        df_model_data = process_data(pd.read_csv(os.path.join(args.filepath, args.filename), sep=";"))
    
        # Shuffle and splitting dataset
        df_shuffled = df_model_data.sample(frac=1, random_state=1729)
        train_end = int(0.7 * len(df_model_data))
        validation_end = int(0.9 * len(df_model_data))
        train_data = df_shuffled.iloc[:train_end]
        validation_data = df_shuffled.iloc[train_end:validation_end]
        test_data = df_shuffled.iloc[validation_end:]
    
        print(f"Data split > train:{train_data.shape} | validation:{validation_data.shape} | test:{test_data.shape}")
        mlflow.log_params(
            {
                "train": train_data.shape,
                "validate": validation_data.shape,
                "test": test_data.shape
            }
        )

        mlflow.set_tags(
            {
                'mlflow.user':user_profile_name,
                'mlflow.source.type':'JOB'
            }
        )
        
        # Save datasets locally
        train_data.to_csv(os.path.join(args.outputpath, 'train/train.csv'), index=False, header=False)
        validation_data.to_csv(os.path.join(args.outputpath, 'validation/validation.csv'), index=False, header=False)
        test_data[target_col].to_csv(os.path.join(args.outputpath, 'test/test_y.csv'), index=False, header=False)
        test_data.drop([target_col], axis=1).to_csv(os.path.join(args.outputpath, 'test/test_x.csv'), index=False, header=False)
        
        # Save the baseline dataset for model monitoring
        df_model_data.drop([target_col], axis=1).to_csv(os.path.join(args.outputpath, 'baseline/baseline.csv'), index=False, header=False)

        mlflow.log_artifact(local_path=os.path.join(args.outputpath, 'baseline/baseline.csv'))

        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "train/train.csv"))
        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "validation/validation.csv"))
        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "test/test_x.csv"))
        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "test/test_y.csv"))
    
    print("## Processing complete. Exiting.")


#### Create an experiment run
Create a new run in your experiment to track parameters, configuration, inputs, and outputs of the processing job.

In [ ]:
run_name = name_from_base("container-processing-", short=True)
run_id = mlflow.start_run(
    run_name=run_name,
    description="feature engineering with a SageMaker processing job (Part 2)").info.run_id

#### Set the job inputs and outputs
Note how SageMaker maps your data to the local paths on the processing container's EBS volume:

![](img/data-processing.png)

The input data can come from a local directory, an S3 bucket, Amazon Athena, or Amazon Redshift. The managed job reads the raw dataset from S3 (`s3_data_uri`, uploaded in notebook 00) and writes the processed datasets back to S3.

First, define the S3 output locations for the data sets. These are stored with `%store` because notebooks 03 and 04 consume them.

In [ ]:
train_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "train")
validation_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "validation")
test_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "test")
baseline_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "baseline")

%store train_s3_url
%store validation_s3_url
%store test_s3_url
%store baseline_s3_url

Next, define the `ProcessingInput` and `ProcessingOutput` objects which help SageMaker understand where to find the input data and where to save the output data for the processing job.

In [ ]:
processing_inputs = [
    ProcessingInput(
        input_name="data",
        s3_input=ProcessingS3Input(
            s3_uri=s3_data_uri,
            local_path="/opt/ml/processing/input",
            s3_data_type="S3Prefix",
            s3_input_mode="File")
    )
]

In [ ]:
processing_outputs = [
    ProcessingOutput(
        output_name="train",
        s3_output=ProcessingS3Output(
            s3_uri=train_s3_url,
            local_path="/opt/ml/processing/output/train",
            s3_upload_mode="EndOfJob")
    ),
    ProcessingOutput(
        output_name="validation",
        s3_output=ProcessingS3Output(
            s3_uri=validation_s3_url,
            local_path="/opt/ml/processing/output/validation",
            s3_upload_mode="EndOfJob")
    ),
    ProcessingOutput(
        output_name="test",
        s3_output=ProcessingS3Output(
            s3_uri=test_s3_url,
            local_path="/opt/ml/processing/output/test",
            s3_upload_mode="EndOfJob")
    ),
    ProcessingOutput(
        output_name="baseline",
        s3_output=ProcessingS3Output(
            s3_uri=baseline_s3_url,
            local_path="/opt/ml/processing/output/baseline",
            s3_upload_mode="EndOfJob")
    )
]

#### Create the processor
Instantiate a [`FrameworkProcessor`](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job-frameworks.html) with the custom BYOC container you built in notebook 00 and an `ml.m5.xlarge` instance.

`FrameworkProcessor` overrides the container entrypoint with a generated `runproc.sh`, so any image that provides `bash`, `tar`, `pip`, `python3`, and the script's dependencies works — no processing-specific entrypoint is needed. The MLflow coordinates are passed as environment variables so the script logs into the run you just opened.

In [ ]:
# Use the custom BYOC image built in notebook 00 (restored via %store -r).
# It satisfies the FrameworkProcessor container contract: bash, tar, pip,
# and python3, plus the sklearn/pandas/mlflow stack used by preprocessing.py.
print(f"Processing image: {xgboost_image}")

processor = FrameworkProcessor(
    image_uri=xgboost_image,
    role=sm_role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    max_runtime_in_seconds=600,  # stop the job (and billing) after 10 minutes
    base_job_name='from-idea-to-prod-processing',
    command=["python3"],
    env={
        'MLFLOW_TRACKING_ARN': mlflow_arn,
        'MLFLOW_EXPERIMENT_NAME': experiment_name,
        'MLFLOW_RUN_ID': run_id,
        'REGION': region,
        'USER': user_profile_name,
    }
)

#### Start the SageMaker processing job
The first time the processor runs it authenticates Docker to ECR and pulls the BYOC image from your account's registry, so this run is slower than subsequent ones.

In [ ]:
processor.run(
    code="preprocessing.py",
    source_dir="./processing",
    requirements="requirements.txt",
    inputs=processing_inputs,
    outputs=processing_outputs,
    wait=False,
    logs=False,
)

# The processing job above was submitted with wait=False, so .run() returned
# immediately without blocking. Poll the processing job status in a loop here
# until it reaches a terminal state.
processing_job = processor.latest_job
print(f"Polling processing job: {processing_job.processing_job_name}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    processing_job.refresh()
    status = processing_job.processing_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    failure_reason = processing_job.failure_reason or ""
    if "MaxRuntimeExceeded" in failure_reason:
        # Rescue path: the job hit the max_runtime_in_seconds cap (600s).
        # SageMaker already stopped the containers and billing. Close the
        # MLflow run with a failed state and give actionable guidance
        # instead of a bare stack trace.
        mlflow.end_run(status="FAILED")
        raise TimeoutError(
            f"Processing job {processing_job.processing_job_name} exceeded its "
            f"maximum runtime of 600 seconds and was stopped by SageMaker "
            f"(FailureReason: {failure_reason}). If the job legitimately needs "
            f"more time, increase max_runtime_in_seconds on the FrameworkProcessor; "
            f"otherwise check the CloudWatch logs of the job for a hang "
            f"(e.g. waiting on a network resource such as the MLflow tracking server)."
        )
    mlflow.end_run(status="FAILED")
    raise RuntimeError(
        f"Processing job {processing_job.processing_job_name} ended with status '{status}': "
        f"{failure_reason}"
    )
print(f"Processing job {processing_job.processing_job_name} completed successfully.")

mlflow.end_run()

Once the job has finished, read the training output back from S3 as a quick sanity check.

In [ ]:
job = processor.latest_job
output_uris = {
    output.output_name: output.s3_output.s3_uri
    for output in job.processing_output_config.outputs
}

# Sanity check on the train channel only — the other outputs are consumed by the
# training job, the endpoint test, and notebooks 03/04 straight from S3.
train_data = pd.read_csv(
    StringIO(S3Downloader.read_file(S3Downloader.list(output_uris["train"])[0])),
    header=None,
)
train_data.head()

## Model training with SageMaker training jobs
You can follow the same approach as with processing and now run the model training as a [SageMaker training job](https://docs.aws.amazon.com/sagemaker/latest/dg/train-model.html). You provide a training image, source code, and compute configuration.

The following picture summarizes the SageMaker training options:

![](img/sagemaker-training-options.png)

First, let's create a folder for our training scripts and create a requirements.txt with MLflow.

In [ ]:
local_training_dir = "./training/"
os.makedirs(local_training_dir, exist_ok=True)
with open(os.path.join(local_training_dir, "requirements.txt"), "w") as file_handle:
    file_handle.write("sagemaker>=3,<4")

### Prepare data and hyperparameters
Define the data input channels for the training job.

In [ ]:
training_input_data = [
    InputData(channel_name='train', data_source=train_s3_url),
    InputData(channel_name='validation', data_source=validation_s3_url),
]

output_s3_url = f"s3://{bucket_name}/{bucket_prefix}/output"
%store output_s3_url

Set the instance type and instance count for the training jobs.

In [ ]:
train_instance_count = 1
train_instance_type = "ml.r5.xlarge"

Prepare the hyperparameters and MLflow settings in environment variables.

In [ ]:
hyperparams = {
    'num_round': 50,
    'max_depth': 3,
    'eta': 0.5,
    'alpha': 2.5,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'early_stopping_rounds': 10,
    'verbosity': 1
}

In [ ]:
env_variables = {
    'MLFLOW_TRACKING_ARN': mlflow_arn,
    'MLFLOW_EXPERIMENT_NAME': experiment_name,
    'USER': user_profile_name,
    'REGION': region,
    'TRAINING_JOB_CONTAINER': xgboost_image
}

print(env_variables)

### Train using the `@remote` decorator
First, we show how to use the [`@remote` decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/train-remote-decorator-invocation.html), which lets you run any Python function as a remote SageMaker training job. This is the simplest way to move local training code to SageMaker managed infrastructure – just annotate your function and call it.

#### Define the training function
This is an ordinary Python function — no SageMaker API inside. You already validated this training logic locally in Part 1 (same algorithm, same data, in this kernel), so there is no separate local invocation here; the function goes straight to a managed job in the next cell.

In [ ]:
def train_xgboost(train_uri, validation_uri, hyperparams):
    """Train an XGBoost model"""
    import os
    import boto3
    import pandas as pd
    import xgboost as xgb
    from sklearn.metrics import roc_auc_score
    import mlflow
    from io import StringIO

    # Download data from S3
    s3 = boto3.client('s3')
    # print(os.environ)
    def read_csv_from_s3(s3_uri):
        bucket = s3_uri.split('/')[2]
        key = '/'.join(s3_uri.split('/')[3:])
        # List objects to find CSV files
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=key)
        csv_files = [o['Key'] for o in resp.get('Contents', []) if o['Key'].endswith('.csv')]
        dfs = []
        for f in csv_files:
            obj = s3.get_object(Bucket=bucket, Key=f)
            dfs.append(pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')), header=None))
        return pd.concat(dfs, ignore_index=True)

    train_df = read_csv_from_s3(train_uri)
    val_df = read_csv_from_s3(validation_uri)

    dtrain = xgb.DMatrix(train_df.iloc[:, 1:], label=train_df.iloc[:, 0])
    dval = xgb.DMatrix(val_df.iloc[:, 1:], label=val_df.iloc[:, 0])

    # Setup MLflow
    mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_ARN'))
    mlflow.set_experiment(experiment_name=os.getenv('MLFLOW_EXPERIMENT_NAME'))
    mlflow.xgboost.autolog(log_model_signatures=True, log_datasets=True, log_models=False)

    train_hp = {k: v for k, v in hyperparams.items() if k not in ('num_round', 'eval_metric', 'early_stopping_rounds', 'alpha', 'colsample_bytree')}

    with mlflow.start_run(run_name='remote-decorator-training', description='xgboost via @remote decorator') as run:
        booster = xgb.train(
            params=train_hp,
            dtrain=dtrain,
            evals=[(dtrain, 'train'), (dval, 'validation')],
            num_boost_round=hyperparams['num_round'],
        )
        val_auc = roc_auc_score(dval.get_label(), booster.predict(dval))
        train_auc = roc_auc_score(dtrain.get_label(), booster.predict(dtrain))
        mlflow.xgboost.log_model(booster, name='xgboost')
        mlflow.log_metrics({'validation_auc': val_auc, 'train_auc': train_auc})
        mlflow.set_tags({
            'mlflow.user': os.environ.get('USER', 'n/a'),
            'mlflow.source.type': 'JOB',
            'mlflow.source.name': os.environ.get('TRAINING_JOB_ARN', 'n/a'),
            'sagemaker.training_container_image': os.environ.get('TRAINING_JOB_CONTAINER', 'n/a'),
            'sagemaker.runtime': 'sagemaker',
            'sagemaker.training_job_arn': os.environ.get('TRAINING_JOB_ARN', 'n/a')
        })
        print(f'train-auc: {train_auc:.4f}, validation-auc: {val_auc:.4f}')

    return booster

In [ ]:
## set ENV variables locally
import os

for k, v in env_variables.items():
    os.environ[k] = v

In [ ]:
booster = train_xgboost(train_s3_url, validation_s3_url, hyperparams)

#### Annotate the function with the `@remote` decorator
To run the function on SageMaker managed infrastructure, annotate it. To avoid redefining the body, decorate a thin wrapper `remote_train_xgboost` with the same signature that calls `train_xgboost`. The decorator arguments describe the job: instance type, role, extra dependencies, environment variables, and the container image.

In [ ]:
@remote(
    instance_type=train_instance_type,
    role=sm_role,
    dependencies='./training/requirements.txt',
    environment_variables=env_variables,
    job_name_prefix='from-idea-to-prod-remote',
    keep_alive_period_in_seconds=3600,
    image_uri=xgboost_image
)
def remote_train_xgboost(train_uri, validation_uri, hyperparams):
    """Train an XGBoost model using the @remote decorator."""
    return train_xgboost(train_uri, validation_uri, hyperparams)

In [ ]:
booster = remote_train_xgboost(train_s3_url, validation_s3_url, hyperparams)

---

The `@remote` decorator is the quickest way to move local code to SageMaker. Next, you'll use `ModelTrainer` which gives you more control over the training container, script packaging, and output artifacts — the same pattern used in SageMaker Pipelines.

### Train using `ModelTrainer` in script mode
The [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/api/training/model_trainer.html) class provides full control over the training job configuration. Use it when you need to run a training script inside a specific training container — a SageMaker built-in image or your own — with custom hyperparameters, input channels, and output locations. We also recommend this when running jobs at scale or including jobs in a pipeline.

In the previous section where we used the `@remote` decorator, SageMaker ran a training job based on our notebook environment and the additional requirements.txt we provided. For running remote training jobs with [ModelTrainer](https://sagemaker.readthedocs.io/en/stable/training/index.html), we will specify a Docker image as the environment. We use the custom BYOC image built in notebook 00 — the same image that just ran the processing job — so processing, training, and inference all share one dependency set. (Alternatively, you could fetch the [pre-built XGBoost container](https://github.com/aws/sagemaker-xgboost-container) here instead.)

In [ ]:
xgb_training_image = xgboost_image
print(f'XGBoost training image: {xgb_training_image}')

#### Prepare the training script
Create a `train.py` file with the training script.

In [ ]:
%%writefile ./training/train.py
import argparse
import json
import os
import pickle as pkl

import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# NOTE: The original script used `get_dmatrix` from `sagemaker_xgboost_container`,
# a module that only exists inside AWS's managed XGBoost container. The BYOC
# image built from `byoc/Dockerfile` does not include it. The local helper
# `load_csv_dmatrix` below is a drop-in replacement that uses only pandas and
# xgboost, so the script works against both the AWS-managed image and the BYOC
# image.
def load_csv_dmatrix(channel_dir):
    """Build an xgb.DMatrix from all CSV files in a SageMaker channel dir.
    Assumes header-less CSVs with the label in column 0 (SageMaker XGBoost convention)."""
    if channel_dir is None or not os.path.isdir(channel_dir):
        return None
    files = sorted(
        os.path.join(channel_dir, f)
        for f in os.listdir(channel_dir)
        if not f.startswith('.') and os.path.isfile(os.path.join(channel_dir, f))
    )
    if not files:
        return None
    frames = [pd.read_csv(f, header=None) for f in files]
    df = pd.concat(frames, ignore_index=True)
    labels = df.iloc[:, 0].values
    features = df.iloc[:, 1:].values
    return xgb.DMatrix(features, label=labels)

from time import gmtime, strftime

suffix = strftime('%d-%H-%M-%S', gmtime())
user_profile_name = os.getenv('USER', 'sagemaker')
experiment_name = os.getenv('MLFLOW_EXPERIMENT_NAME')
region = os.getenv('REGION')


def _xgb_train(params, dtrain, dval, evals, num_boost_round, model_dir, is_master):
    booster = xgb.train(params=params, dtrain=dtrain, evals=evals, num_boost_round=num_boost_round)

    val_auc = roc_auc_score(dval.get_label(), booster.predict(dval))
    train_auc = roc_auc_score(dtrain.get_label(), booster.predict(dtrain))
    mlflow.log_params(params)
    mlflow.log_metrics({'validation_auc': val_auc, 'train_auc': train_auc})
    print(f'[0]#011train-auc:{train_auc}#011validation-auc:{val_auc}')

    if is_master:
        model_location = model_dir + '/xgboost-model'
        pkl.dump(booster, open(model_location, 'wb'))
        print(f'Stored trained model at {model_location}')
    
    return booster


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--max_depth', type=int)
    parser.add_argument('--eta', type=float)
    parser.add_argument('--alpha', type=float)
    parser.add_argument('--gamma', type=int)
    parser.add_argument('--min_child_weight', type=float)
    parser.add_argument('--subsample', type=float)
    parser.add_argument('--colsample_bytree', type=float)
    parser.add_argument('--verbosity', type=int)
    parser.add_argument('--objective', type=str)
    parser.add_argument('--num_round', type=int)
    parser.add_argument('--early_stopping_rounds', type=int)
    parser.add_argument('--tree_method', type=str, default='auto')
    parser.add_argument('--predictor', type=str, default='auto')
    parser.add_argument('--output_data_dir', type=str, default=os.environ.get('SM_OUTPUT_DATA_DIR'))
    parser.add_argument('--model_dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--validation', type=str, default=os.environ.get('SM_CHANNEL_VALIDATION'))
    parser.add_argument('--sm_hosts', type=str, default=os.environ.get('SM_HOSTS'))
    parser.add_argument('--sm_current_host', type=str, default=os.environ.get('SM_CURRENT_HOST'))
    parser.add_argument('--sm_training_env', type=str, default=os.environ.get('SM_TRAINING_ENV'))

    args, _ = parser.parse_known_args()

    sm_hosts = json.loads(args.sm_hosts)
    sm_current_host = args.sm_current_host
    dtrain = load_csv_dmatrix(args.train)
    dval = load_csv_dmatrix(args.validation)
    watchlist = [(dtrain, 'train'), (dval, 'validation')] if dval is not None else [(dtrain, 'train')]
    sm_training_env = json.loads(args.sm_training_env)

    import mlflow
    mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_ARN'))
    mlflow.set_experiment(experiment_name=experiment_name if experiment_name else f'train-{suffix}')
    mlflow.xgboost.autolog(log_model_signatures=True, log_datasets=True, log_models=False)

    train_hp = {
        'max_depth': args.max_depth, 'eta': args.eta, 'gamma': args.gamma,
        'min_child_weight': args.min_child_weight, 'subsample': args.subsample,
        'verbosity': args.verbosity, 'objective': args.objective,
        'tree_method': args.tree_method, 'predictor': args.predictor,
    }

    xgb_train_args = dict(
        params=train_hp, dtrain=dtrain, dval=dval, evals=watchlist,
        num_boost_round=args.num_round, model_dir=args.model_dir,
    )

    # Resume parent run if MLFLOW_RUN_ID is set (for nested run grouping)
    parent_run_id = os.getenv('MLFLOW_RUN_ID')
    if parent_run_id:
        mlflow.start_run(run_id=parent_run_id)

    with mlflow.start_run(
        run_name=f'container-training-{suffix}',
        description='xgboost running in SageMaker container in script mode',
        nested=True if parent_run_id else False,
    ) as run:
        mlflow.set_tags({
            'mlflow.user': user_profile_name,
            'mlflow.source.type': 'JOB',
            'mlflow.source.name': os.environ.get('TRAINING_JOB_ARN', 'n/a'),
            'sagemaker.training_container_image': os.environ.get('TRAINING_JOB_CONTAINER', 'n/a'),
            'sagemaker.job_name': sm_training_env['job_name'],
            'sagemaker.runtime': 'sagemaker',
            'sagemaker.training_job_arn': os.environ.get('TRAINING_JOB_ARN', 'n/a')
        })
        if dtrain:
            xgb_train_args['is_master'] = True
            booster = _xgb_train(**xgb_train_args)
            mlflow.xgboost.log_model(booster, name=f'xgboost-{suffix}')
        else:
            raise ValueError('Training channel must have data to train model.')


#### Train with `ModelTrainer` in local container mode
`ModelTrainer` supports two `training_mode` values: `LOCAL_CONTAINER` and `SAGEMAKER_TRAINING_JOB`. This is the one place in the notebook where you use **local mode**: the training script runs inside the BYOC image in a Docker container on this space, which validates the container contract (`SM_CHANNEL_*` paths, `SM_MODEL_DIR`, hyperparameter parsing, installed dependencies) in seconds instead of waiting on a managed instance.

In [ ]:
train_locally = ModelTrainer(
    training_image=xgb_training_image,
    source_code=SourceCode(
        source_dir="./training",
        entry_script="train.py",
    ),
    compute=Compute(instance_type="local", instance_count=1),
    training_mode=TrainingMode.LOCAL_CONTAINER,
    hyperparameters=hyperparams,
    base_job_name='from-idea-to-prod-local',
    environment=env_variables,
)

<div class="alert alert-info">If you get an error <code>No such file or directory: 'docker'</code> in the code below, you need to run the section <b>Install Docker to enable Studio local mode</b> from the notebook 00.</div>

In [ ]:
try:
    train_locally.train(
        input_data_config=[
            InputData(channel_name="train", data_source=train_s3_url, content_type="text/csv"),
            InputData(channel_name="validation", data_source=validation_s3_url, content_type="text/csv"),
        ],
        wait=True,
        logs=True,
    )
except PermissionError as e:
    if "__pycache__" in str(e):
        print(f"⚠️ Training succeeded but cleanup failed (harmless): {e}")
    else:
        raise

#### Train on SageMaker managed infrastructure
The only changes are the compute configuration, the `SAGEMAKER_TRAINING_JOB` mode, and an explicit output location. The script is unchanged.

In [ ]:
trainer_managed = ModelTrainer(
    training_image=xgb_training_image,
    source_code=SourceCode(
        source_dir="./training",
        entry_script="train.py",
    ),
    compute=Compute(
        instance_type=train_instance_type,
        instance_count=train_instance_count
    ),
    hyperparameters=hyperparams,
    base_job_name='from-idea-to-prod-training',
    training_mode=TrainingMode.SAGEMAKER_TRAINING_JOB,
    output_data_config=OutputDataConfig(s3_output_path=output_s3_url),
    environment=env_variables,
)

In [ ]:
trainer_managed.train(
    input_data_config=training_input_data,
    wait=False,
    logs=False,
)

In [ ]:
# The training job above was submitted with wait=False, so .train() returned
# immediately without blocking. Poll the training job status in a loop here
# until it reaches a terminal state.
training_job = trainer_managed._latest_training_job
print(f"Polling training job: {training_job.training_job_name}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    training_job.refresh()
    status = training_job.training_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status} | secondary={training_job.secondary_status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    raise RuntimeError(
        f"Training job {training_job.training_job_name} ended with status '{status}': "
        f"{training_job.failure_reason}"
    )
print(f"Training job {training_job.training_job_name} completed successfully.")

After the training job completes, you can fetch the output artifacts.

In [ ]:
training_job = trainer_managed._latest_training_job
print(f"Training job: {training_job.training_job_name}")
print(f"Model artifacts: {training_job.model_artifacts.s3_model_artifacts}")

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://{}.console.aws.amazon.com/sagemaker/home?region={}#/jobs/{}">training job</a></b>'.format(
            region, region, trainer_managed._latest_training_job.training_job_name))
)

## Select the best candidate model

By now the experiment holds several **candidate** models, each logged by a SageMaker job:

| Candidate | Logged by |
|---|---|
| `remote-decorator-training` | the `@remote` training job |
| `container-training-…` | the `ModelTrainer` run in `LOCAL_CONTAINER` mode |
| `container-training-…` | the `ModelTrainer` managed training job |

Rather than hard-coding which one to register, ask the tracking server which performed best. [`mlflow.search_logged_models()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.search_logged_models) queries **logged models** directly and returns MLflow 3 `LoggedModel` entities, each already carrying its `model_id`, `artifact_location`, `source_run_id`, and metrics — so there is no run-to-model resolution step.

`training/train.py` runs `mlflow.xgboost.autolog(..., log_models=False)` and then logs the model explicitly with `mlflow.xgboost.log_model(booster, name="xgboost")`, which is what makes it a searchable logged model with the run's metrics attached.

Two details worth knowing:

- Ordering by `metrics.validation_auc` only considers models that *have* that metric, so the Part 1 prototypes are excluded automatically — they logged `test_auc`. The explicit `filter_string` states that requirement rather than relying on it.
- The two `container-training` candidates run the same script on the same data, so their scores are effectively identical and the tie is broken arbitrarily. Both are genuine MLflow models in the artifact store and both deploy the same way. To restrict the search to managed training jobs only, add a tag filter — `train.py` sets `sagemaker.job_name` on its runs.

The ranked table below shows exactly which candidate won.

In [ ]:
mlflow_client = MlflowClient()

metric_name = "validation_auc"
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

candidates = mlflow.search_logged_models(
    experiment_ids=[experiment_id],
    filter_string=f"metrics.{metric_name} > 0",
    order_by=[{"field_name": f"metrics.{metric_name}", "ascending": False}],
    output_format="list",
)

if not candidates:
    raise RuntimeError(
        f"No logged model in experiment '{experiment_name}' has a '{metric_name}' metric. "
        "Did the training jobs above complete and log their models?"
    )


def metric_of(model, key):
    return {m.key: m.value for m in (model.metrics or [])}.get(key)


print(f"{'rank':<5}{'run name':<34}{metric_name:>12}   model id")
print("-" * 100)
for rank, candidate in enumerate(candidates, 1):
    run_name = mlflow_client.get_run(candidate.source_run_id).info.run_name
    print(f"{rank:<5}{run_name:<34}{metric_of(candidate, metric_name):>12.4f}   {candidate.model_id}")

logged_model = candidates[0]
best_run_id = logged_model.source_run_id

print(f"\nSelected model id: {logged_model.model_id}")
print(f"Source run:        {best_run_id}")
print(f"Artifact location: {logged_model.artifact_location}")
print(f"{metric_name}:     {metric_of(logged_model, metric_name):.4f}")

## Explain the model before you register it

You have a winner on paper - one AUC number. That is not enough to put a model in front of customers, and it is not enough to justify approving it in a registry. Before registration, produce a **feature-attribution and fairness report** for exactly the model you are about to register.

Doing it *here*, before `register_model()`, is deliberate: the report is logged onto the same MLflow run as the model, so the Model Package inherits lineage that already includes the explanation, and the approval step later becomes an informed decision rather than a rubber stamp.

### What computes the explanation

<div class="alert alert-info">
<b>Note on Amazon SageMaker Clarify:</b> new customer access to Clarify closed on 2026-07-30 and no new features are planned (see the <a target="_blank" href="https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-availability-change.html">availability change guidance</a>). This workshop uses the building blocks Clarify was made of: <b>SHAP</b> for feature attribution - the library Clarify used internally - and the published, standardized <b>bias-metric formulas</b> computed directly with pandas.
</div>

The logic lives in [`pipeline_steps/explain.py`](pipeline_steps/explain.py) as a single plain Python function, `explain(...)`. That placement is the point: you run it here as a **managed job** by applying the `@remote` decorator to it, and notebook 03 wraps the *same* function with `@step` to make it a pipeline step. One implementation, two decorators - the same `@remote` to `@step` progression this workshop teaches for training.

The function uses `shap.TreeExplainer` rather than the `KernelExplainer` you may have seen elsewhere. Kernel SHAP is model-agnostic and is what you need to explain an opaque pipeline; our model is a plain XGBoost booster, so tree SHAP applies and is **exact** - no sampling, and fast enough to explain the entire test set instead of a subsample.

### Prepare the test data
Download the held-out test set the processing job produced. It is used three times from here on: to check the feature names line up, by the endpoint scoring, and by the confusion matrix.

In [ ]:
!aws s3 cp $test_s3_url/test_x.csv tmp/test_x.csv
!aws s3 cp $test_s3_url/test_y.csv tmp/test_y.csv

Load it into memory. The explainability job reads the same files straight from S3 - these local copies are for the checks and the scoring in this notebook.

In [ ]:
test_x = np.loadtxt("tmp/test_x.csv", delimiter=",")
test_y = np.loadtxt("tmp/test_y.csv", delimiter=",")

print(f"test_x: {test_x.shape} | test_y: {test_y.shape}")

### Feature names and the fairness facet

The processing job wrote header-less CSVs, so the feature names are not in `test_x.csv`. They come from the dataframe you built in Part 1 - the same one-hot columns the job produced, in the same order - and travel to the job as an argument.

For the fairness analysis you need a **facet**: a group whose treatment you want to compare against everyone else. Part 1's age binning left us one-hot columns per age band, so `age_60-69` and `age_70-plus` together express *customer aged 60 or over* - a legitimate question for a term-deposit marketing model.

In [ ]:
# Part 1 built df_model_data with the target first, then the one-hot features, and
# preprocessing.py applied the identical transformation before writing test_x.csv.
feature_names = [c for c in df_model_data.columns if c != target_col]

facet_columns = ["age_60-69", "age_70-plus"]

assert len(feature_names) == test_x.shape[1], (
    f"{len(feature_names)} feature names vs {test_x.shape[1]} columns in test_x.csv"
)
missing_facets = [c for c in facet_columns if c not in feature_names]
assert not missing_facets, f"facet columns not in the feature set: {missing_facets}"

print(f"{len(feature_names)} features, e.g. {feature_names[:6]}")
print(f"Fairness facet: {facet_columns}")

### Run the report as a managed job

Apply `@remote` to the imported `explain` function. Two arguments matter beyond the usual compute settings:

- **`dependencies`** - the BYOC image ships xgboost, pandas, mlflow and s3fs, but not `shap` or `matplotlib`. [`explainability/requirements.txt`](explainability/requirements.txt) adds just those two, and the decorator pip-installs them into the job.
- **`include_local_workdir`** - `explain` lives in a module rather than this notebook, so the job needs the `pipeline_steps/` directory shipped alongside it. `CustomFileFilter` keeps the upload small by excluding the datasets, notebooks and images.

Note the environment variables: `explain` follows the same convention as the pipeline steps in notebook 03 and lets MLflow pick up `MLFLOW_TRACKING_URI` and `MLFLOW_EXPERIMENT_NAME` natively from the environment, rather than calling `set_tracking_uri` itself.

In [ ]:
explain_env = {
    # MLflow reads these two natively from the environment.
    "MLFLOW_TRACKING_URI": mlflow_arn,
    "MLFLOW_EXPERIMENT_NAME": experiment_name,
    # picked up by pipeline_steps/runtime_utils.log_runtime_info()
    "CONTAINER_IMAGE": xgboost_image,
    "USER": user_profile_name,
    "REGION": region,
}

# Applying the decorator as a call, rather than with @-syntax, keeps the shared
# function untouched - and reads exactly like step(explain, ...) does in notebook 03.
remote_explain = remote(
    instance_type=train_instance_type,
    role=sm_role,
    image_uri=xgboost_image,
    dependencies="./explainability/requirements.txt",
    include_local_workdir=True,
    custom_file_filter=CustomFileFilter(
        ignore_name_patterns=["data", "tmp", "img", "*.ipynb", "__pycache__", ".git"]
    ),
    environment_variables=explain_env,
    job_name_prefix="from-idea-to-prod-explain",
    keep_alive_period_in_seconds=3600,
)(explain)

explain_result = remote_explain(
    mlflow_run_id=best_run_id,
    test_x_data_s3_path=f"{test_s3_url}/test_x.csv",
    test_y_data_s3_path=f"{test_s3_url}/test_y.csv",
    feature_names=feature_names,
    output_s3_prefix=f"s3://{bucket_name}/{bucket_prefix}",
    facet_columns=facet_columns,
)

### Read the report

The job logged the plots, the per-instance SHAP table and the bias metrics to the same MLflow run as the model, and returned the headline numbers. Open the run in MLflow to see the beeswarm, the global importance bar chart and the waterfall for the most confident prediction.

Read `DI_disparate_impact` as the ratio of positive-prediction rates between the facet and the comparison group: `1.0` is parity, and the further from `1.0`, the more the model treats the two groups differently. `DPL` versus `DPPL` tells you whether a gap was already in the labels (pre-training) or was introduced by the model (post-training) - a `DPPL` close to `DPL` means the model inherited the imbalance rather than amplifying it.

Check `n_facet_group` before trusting either ratio: on this dataset the age-60+ facet is only a few dozen of the ~4,100 test rows, so its rates move a lot with a handful of predictions. A large `DI` on a small facet is a prompt to collect more data for that group, not yet evidence of a broken model.

In [ ]:
print(f"Explained model:  {explain_result['explained_model_id']}")
print(f"E[y] (log-odds):  {explain_result['shap_expected_value_logodds']:.4f}")
print(f"Report in S3:     {explain_result['report_s3_prefix']}")

print("\nTop features by mean |SHAP|:")
for rank, feature in enumerate(explain_result["top_features"][:10], 1):
    print(f"  {rank:<3}{feature}")

print("\nFairness metrics for the age-60+ facet:")
for name, value in explain_result["bias_metrics"].items():
    print(f"  {name:<40}{value:.4f}" if isinstance(value, float) else f"  {name:<40}{value}")

## Explore the explainability results

In [ ]:
explain_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment_id}/runs/{explain_result['explainability_run_id']}?workspace=default"
)
display(Javascript('window.open("{}");'.format(explain_run_link)))

## Explore the explainability artifacts

In [ ]:
explain_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment_id}/runs/{explain_result['explainability_run_id']}/artifacts?workspace=default"
)
display(Javascript('window.open("{}");'.format(explain_run_link)))

### Make the model deployable, then register it

The training jobs logged their models into the MLflow artifact store. Registering one is a single call — but a registered model is only useful if it can actually be *served*. That is what the **inference specification logging** pattern gives you:

1. Upload an `inference.py` into that logged model's artifact store under `code/`, using `MlflowClient.log_model_artifacts()`.
2. Log an **inference specification** on the logged model with the `sagemaker-mlflow` plugin (`>= 0.5.0`): the serving container image, an `S3Prefix` `ModelDataSource` pointing at the MLflow artifact location, and the `SAGEMAKER_PROGRAM` / `SAGEMAKER_SUBMIT_DIRECTORY` environment variables.
3. Call `mlflow.register_model()`.

```
MLflow artifact store ◀────────── serves directly from ──────────┐
        │  log_inference_specification()                         │
        ▼                                                        │
MLflow ──register──▶ auto-sync ──▶ Model Package (deployable) ──▶ Endpoint
```

The MLflow artifact store stays the single source of truth for the model bytes: no repacking, no artifact copies, and no `update_model_package` call to patch the specification in afterwards.

<div class="alert alert-warning">
<b>Order matters:</b> the specification must be logged <b>before</b> <code>mlflow.register_model()</code>. The auto-sync copies the specification onto the Model Package at registration time — log it afterwards and the package is already created without it.
</div>

These steps are exactly what the reusable [`pipeline_steps/register.py`](pipeline_steps/register.py) step does in notebook 03 — it receives the run id directly from its training step instead of searching for it. Here you do it inline, so you can see each piece.

#### Log an `inference.py` into the model's artifact store

The BYOC image serves with its own FastAPI stack (see [`byoc/serving_app.py`](byoc/serving_app.py)), which loads the script named by `SAGEMAKER_PROGRAM` and dispatches requests to `model_fn` / `input_fn` / `predict_fn` / `output_fn`. It provides no default model loader, so a `model_fn` is mandatory — without it the container fails every `/ping` health check and the endpoint never reaches `InService`.

The `S3Prefix` `ModelDataSource` in the specification downloads *everything* under the logged model's artifact location to `/opt/ml/model`, so a file logged at `code/inference.py` arrives at `/opt/ml/model/code/inference.py`, and the MLflow model files (`MLmodel`, the booster, `conda.yaml`) land at `/opt/ml/model` — which is why `model_fn` can just call `mlflow.xgboost.load_model(model_dir)`.

Logging the script through MLflow rather than with a raw `s3:PutObject` keeps the artifact store as the single source of truth, and it shows up in the MLflow UI next to the model.

In [ ]:
INFERENCE_SCRIPT = r"""
import json

import numpy as np


def model_fn(model_dir):
    # The S3Prefix ModelDataSource downloaded the whole MLflow model directory
    # here, so MLflow can load the booster and resolve its serialization format.
    import mlflow.xgboost

    return mlflow.xgboost.load_model(model_dir)


def input_fn(request_body, request_content_type):
    if isinstance(request_body, (bytes, bytearray)):
        request_body = request_body.decode("utf-8")
    content_type = (request_content_type or "").split(";")[0].strip()

    if content_type == "text/csv":
        # One record per line, comma-separated features, no header and no label
        # column — the format preprocessing.py wrote for test_x.csv.
        rows = [r for r in request_body.strip().split("\n") if r]
        return np.array([[float(v) for v in r.split(",")] for r in rows])

    if content_type == "application/json":
        payload = json.loads(request_body)
        if isinstance(payload, dict):
            payload = payload.get("instances", payload.get("inputs", payload))
        array = np.array(payload, dtype=float)
        return array.reshape(1, -1) if array.ndim == 1 else array

    raise ValueError(f"Unsupported content type: {content_type}")


def predict_fn(input_data, model):
    import xgboost as xgb

    return model.predict(xgb.DMatrix(input_data))


def output_fn(prediction, accept):
    accept = (accept or "text/csv").split(";")[0].strip()
    values = [float(p) for p in np.atleast_1d(prediction)]
    if accept == "application/json":
        return json.dumps(values), "application/json"
    return "\n".join(str(v) for v in values), "text/csv"
"""

# log_model_artifacts preserves the local directory layout, so code/inference.py
# lands at <artifact_location>/code/inference.py.
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "code"))
    with open(os.path.join(tmp, "code", "inference.py"), "w") as f:
        f.write(INFERENCE_SCRIPT)
    mlflow_client.log_model_artifacts(logged_model.model_id, tmp)

print(f"Logged code/inference.py to {logged_model.artifact_location}/code/")

#### Log the inference specification

`sagemaker_mlflow.log_inference_specification()` attaches the container image, the model data source, and the environment variables to the logged model. The instance types you list here constrain what the Model Package can be deployed on, so include the type you intend to use.

In [ ]:
inference_spec = {
    "Containers": [{
        "Image": xgb_training_image,
        # Serve straight from the MLflow artifact store — no repacking, no copies.
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": logged_model.artifact_location.rstrip("/") + "/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        # Point the BYOC serving stack at the script logged above.
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    }],
    "SupportedContentTypes": ["text/csv", "application/json"],
    "SupportedResponseMIMETypes": ["text/csv", "application/json"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge", "ml.r5.xlarge"],
    "SupportedTransformInstanceTypes": ["ml.m5.large", "ml.m5.xlarge"],
}

sagemaker_mlflow.log_inference_specification(
    logged_model.model_id, inference_specification=inference_spec
)
print("Inference specification logged on the MLflow model.")

In [ ]:
logged_model_artifacts = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/models/{logged_model.model_id}/artifacts?workspace=default"

In [ ]:
display(Javascript('window.open("{}");'.format(logged_model_artifacts)))

#### Register, and watch the auto-sync

Now register the logged model. This is the only registration in the notebook, so it becomes version 1 of `registered_model_name`.

The sync to the SageMaker Model Registry runs asynchronously, and it records the resulting Model Package ARN as a `sagemaker.model_package_arn` **tag on the MLflow model version** — which is a much simpler way to find the package than searching the registry.

In [ ]:
registered_model_version = mlflow.register_model(
    f"models:/{logged_model.model_id}", registered_model_name
)
print(f"Registered in MLflow: {registered_model_version.name} v{registered_model_version.version}")

model_package_arn = None
for _ in range(20):
    mv = mlflow_client.get_model_version(
        registered_model_version.name, registered_model_version.version
    )
    model_package_arn = mv.tags.get("sagemaker.model_package_arn")
    if model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)

if not model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within the timeout")

print(f"\nSynced SageMaker Model Package: {model_package_arn}")

#### Inspect the auto-created Model Package

This is the object notebooks 03 and 04 build on, so it is worth looking at. It already carries an `InferenceSpecification` — that is the payoff of logging the spec before registering.

<div class="alert alert-warning">
<b>SDK gap worth knowing about:</b> the typed <code>ModelPackage</code> resource deserializes the nested containers into a <code>ContainersItem</code> shape that models only 6 of the API's 16 container fields. <code>Environment</code>, <code>ImageDigest</code>, <code>ContainerHostname</code>, <code>Framework</code> and others are dropped on read, and touching <code>container.environment</code> raises <code>AttributeError</code> — even though the values <i>are</i> stored in the registry. So the cell below reads the fields the shape does model from the typed resource, and falls back to the raw <code>DescribeModelPackage</code> response for the rest.
</div>

In [ ]:
model_package = ModelPackage.get(model_package_name=model_package_arn)

# DescribeModelPackage does not return ModelPackageName for versioned packages,
# which breaks refresh()/update(). The API accepts an ARN in that field, so
# backfill it here.
model_package.model_package_name = model_package.model_package_arn

# The typed ContainersItem shape omits Environment, ImageDigest and friends, so
# read the raw response for those. One extra API call, no guessing.
described = boto3.client("sagemaker").describe_model_package(
    ModelPackageName=model_package_arn
)
raw_container = described["InferenceSpecification"]["Containers"][0]

print(f"Model package group:  {model_package.model_package_group_name}")
print(f"Version:              {model_package.model_package_version}")
print(f"Status:               {model_package.model_package_status}")
print(f"Approval status:      {described.get('ModelApprovalStatus', '(not set)')}")
print(f"Has inference spec:   {model_package.inference_specification is not None}")

print("\nInference specification, as stored in the registry")
print(f"  image:              {raw_container['Image']}")
print(f"  image digest:       {raw_container.get('ImageDigest', '(none)')}")
print(f"  model data source:  {raw_container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"  compression:        {raw_container['ModelDataSource']['S3DataSource']['CompressionType']}")
print(f"  environment:        {raw_container.get('Environment', {})}")
print(f"  content types:      {described['InferenceSpecification']['SupportedContentTypes']}")
print(f"  instance types:     {described['InferenceSpecification']['SupportedRealtimeInferenceInstanceTypes']}")

# Fail loudly if the environment we logged did not survive the sync: without
# SAGEMAKER_PROGRAM a framework container has no inference code to run.
expected_env = inference_spec["Containers"][0]["Environment"]
synced_env = raw_container.get("Environment", {})
mismatched = {k: (v, synced_env.get(k)) for k, v in expected_env.items() if synced_env.get(k) != v}
if mismatched:
    raise RuntimeError(
        f"Environment did not round-trip through the MLflow sync: {mismatched}. "
        "The endpoint would fall back to the container's defaults."
    )
print("\nEnvironment round-tripped through the sync intact.")

The sync also generated a **draft Model Card** on the package, pre-filled with the training metrics and hyperparameters of the winning run. You did not ask for it; it comes with the auto-registration.

In [ ]:
import json

card = json.loads(described["ModelCard"]["ModelCardContent"])
training = card["training_details"]["training_job_details"]

print(f"Model card status: {described['ModelCard']['ModelCardStatus']}")
print("\nTraining metrics recorded on the card:")
for m in training["training_metrics"]:
    print(f"  {m['name']:<22}{m['value']:.4f}")
print("\nHyperparameters recorded on the card:")
for h in training["hyper_parameters"]:
    if h["value"] != "None":
        print(f"  {h['name']:<22}{h['value']}")

## Validate, approve, and deploy from the Model Registry

The Model Package exists but is not approved yet. That ordering is the point of a registry: you get a reviewable artifact first, and deployment is a separate, deliberate step.

So the rest of the notebook does three things in order:

1. **Validate** the model's serving code in this kernel, by importing the very `inference.py` the endpoint will run — no Docker, no endpoint, no cost.
2. **Approve** the Model Package version.
3. **Deploy the approved package version** to a real-time endpoint with the SDK v3 typed resources `Model`, `EndpointConfig`, and `Endpoint`, then score the test set against it.

### Validate the real inference code in this kernel

Before approving anything, check that the code the endpoint will run actually works. The `S3Prefix` `ModelDataSource` in the specification downloads the whole logged-model directory to `/opt/ml/model` on the endpoint, and `byoc/serving_app.py` imports `code/inference.py` from it and calls `model_fn(/opt/ml/model)` once per worker at startup.

You can reproduce that faithfully here: download the same artifacts, import the same script the same way, and drive the same four handlers. No Docker, no endpoint, no cost — and unlike a stand-in `InferenceSpec`, this exercises the exact code path that will serve traffic. A failure here is a failure the endpoint would have surfaced as a `/ping` timeout after several minutes of provisioning.

In [ ]:
local_model_dir = mlflow.artifacts.download_artifacts(
    artifact_uri=f"models:/{logged_model.model_id}"
)

print(f"Model artifacts downloaded to: {local_model_dir}")
print(f"Contents:                      {sorted(os.listdir(local_model_dir))}")
print(f"code/:                         {sorted(os.listdir(os.path.join(local_model_dir, 'code')))}")

In [ ]:
# byoc/serving_app.py resolves the script via SAGEMAKER_PROGRAM and loads it with
# importlib, then calls model_fn(MODEL_DIR) at worker startup. Same thing here.
script_path = os.path.join(local_model_dir, "code", "inference.py")

spec = importlib.util.spec_from_file_location("user_inference_module", script_path)
inference_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_module)

local_model = inference_module.model_fn(local_model_dir)
print(f"model_fn returned: {type(local_model).__name__}")

In [ ]:
# The serving app hands input_fn the raw request body as bytes.
csv_payload = "\n".join(",".join(map(str, row)) for row in test_x).encode("utf-8")

data = inference_module.input_fn(csv_payload, "text/csv")
prediction = inference_module.predict_fn(data, local_model)
body, response_content_type = inference_module.output_fn(prediction, "text/csv")

local_predictions = np.array(body.strip().split("\n"), dtype=float)

print(f"Parsed input shape:    {data.shape}")
print(f"Response content type: {response_content_type}")
print(f"Local test AUC:        {roc_auc_score(test_y, local_predictions):.4f}")

### Approve the Model Package version

Note from the cell above that the auto-synced package has **no** approval status at all — the sync does not default it to `PendingManualApproval`. The call below therefore sets it for the first time.

In a real ModelOps setup this is where a human (or an automated quality gate) signs off, and the approval event can trigger a deployment pipeline. Notebook 03 automates the same decision with a `ConditionStep` on the evaluation AUC, registering as `PendingManualApproval` when the threshold is not met.

In [ ]:
model_package.update(model_approval_status="Approved")
model_package.refresh()

print(f"Approval status: {model_package.model_approval_status}")

### Deploy the approved Model Package version

Deploying from the registry needs no image, no script, and no artifact URI at the call site: all of that is in the package's `InferenceSpecification`. A `ContainerDefinition` that names the Model Package is enough.

The three v3 resources map one-to-one to the SageMaker API objects: a `Model` (what to serve), an `EndpointConfig` (how much of it, on what hardware), and an `Endpoint` (the running HTTPS service).

<div class="alert alert-info">
This endpoint has no authentication of its own — access is controlled entirely by IAM (<code>sagemaker:InvokeEndpoint</code>) and it is not exposed to the public internet. Keep it that way: do not front it with an unauthenticated proxy, and delete it in the clean-up section when you are done.
</div>

In [ ]:
inference_instance_type = "ml.m5.xlarge"
resource_name = f"from-idea-to-prod-mp-{time.strftime('%d%H%M%S', time.gmtime())}"

deployed_model = Model.create(
    model_name=resource_name,
    primary_container=ContainerDefinition(model_package_name=model_package_arn),
    execution_role_arn=sm_role,
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=resource_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=resource_name,
            initial_instance_count=1,
            instance_type=inference_instance_type,
        )
    ],
)

endpoint = Endpoint.create(
    endpoint_name=resource_name,
    endpoint_config_name=resource_name,
)

endpoint_name = endpoint.endpoint_name
print(f"Creating endpoint {endpoint_name} from {model_package_arn}")
print("Poll its status in the next cell (this takes a few minutes).")

In [ ]:
# Endpoint.create() returns as soon as the request is accepted, so poll until the
# endpoint reaches a terminal state.
sm_client = boto3.client("sagemaker")
print(f"Polling endpoint: {endpoint_name}")

terminal_states = {"InService", "Failed"}
while True:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': "
        f"{desc.get('FailureReason')}"
    )
print(f"Endpoint {endpoint_name} is InService.")

# Re-obtain the resource so it carries the final state for .invoke()
endpoint = Endpoint.get(endpoint_name=endpoint_name)

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://{}.console.aws.amazon.com/sagemaker/home?region={}#/endpoints/{}">inference endpoint</a></b>'.format(
            region, region, endpoint_name))
)

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/inference-experience/endpoints/details/{}">inference endpoint</a></b>'.format(
            domain_id, region, endpoint_name))
)

## Explore the full lineage of the model version

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/registry/registered-models/{}/versions/version-{}/lineage">the model package version lineage</a> in the Studio UI</b>'.format(
            domain_id, region, model_package.model_package_group_name, model_package.model_package_version))
)

![](img/model-version-lineage.png)

#### Test endpoint prediction

Once the endpoint is up and running, verify it works by invoking it with a sample payload.

In [ ]:
csv_data = "\n".join(",".join(map(str, row)) for row in test_x)

response = endpoint.invoke(
    body=csv_data,
    content_type="text/csv",
    accept="text/csv",
)
predictions = np.array(
    response.body.read().decode("utf-8").strip().split("\n"), dtype=float
).squeeze()
predictions

In [ ]:
pd.crosstab(
    index=test_y,
    columns=np.round(predictions), 
    rownames=['actuals'], 
    colnames=['predictions']
)

In [ ]:
test_auc = roc_auc_score(test_y, predictions)
print(f"Test-auc: {test_auc:.4f}")

#### Save charts to the MLflow run

It is a good idea to save the confusion matrices from the test data in MLflow for reference.

In [ ]:
def plot_confusion_matrix(
    cm, class_names, title="Confusion matrix", cmap=plt.cm.Blues, normalize=False
):
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        ylim=(cm.shape[0] - 0.5, -0.5),
        xticklabels=class_names,
        yticklabels=class_names,
        title=title,
        ylabel="Ground truth label",
        xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    fmt = ".2f"
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax, fig

In [ ]:
class_names = ["no", "yes"]
confusion_matrix = metrics.confusion_matrix(test_y, np.round(predictions))
ax, fig = plot_confusion_matrix(confusion_matrix, class_names)

In [ ]:
print(
    f"Log confusion matrix to the model {registered_model_version.name} "
    f"version {registered_model_version.version}"
)
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_figure(fig, "confusion_matrix.png")

## TODO: Open the run and the artifacts



## Clean-up
An `InService` endpoint bills per instance-hour whether or not you call it, so delete it. The `Model` and `EndpointConfig` objects cost nothing but are removed too, to keep the account tidy.

The registered model, the Model Package, and the S3 datasets are **kept** — notebooks 03 and 04 build on them.

In [ ]:
for label, resource in (
    ("endpoint", endpoint),
    ("endpoint config", endpoint_config),
    ("model", deployed_model),
):
    try:
        resource.delete()
        print(f"Deleted {label}: {resource_name}")
    except Exception as e:
        print(f"Could not delete {label} {resource_name}: {e}")

# Catch any endpoint left over from an earlier run of this notebook.
sm_client = boto3.client("sagemaker")
for ep in sm_client.list_endpoints()["Endpoints"]:
    ep_name = ep["EndpointName"]
    if ep_name != endpoint_name and ep_name.startswith("from-idea-to-prod"):
        try:
            print(f"Deleting leftover endpoint: {ep_name}")
            sm_client.delete_endpoint(EndpointName=ep_name)
        except Exception as e:
            print(f"Error deleting {ep_name}: {e}")

print("Cleanup complete.")

## Summary

In this notebook you:

**Part 1 — in the notebook**
- Explored and visualized the bank marketing dataset
- Engineered features (indicator columns, binning, scaling, one-hot encoding)
- Trained XGBoost locally with cross-validation and a `max_depth` sweep
- Tracked runs, parameters, metrics, and artifacts in managed MLflow — and registered nothing, because a kernel-trained prototype is not a deployment candidate

**Part 2 — on SageMaker AI managed infrastructure**
- **Processing**: moved the same feature engineering into `preprocessing.py` and ran it once as a `FrameworkProcessor` job, producing the canonical train/validation/test/baseline datasets in S3
- **Training with `@remote`**: turned a plain Python function into a SageMaker training job with a single decorator
- **Training with `ModelTrainer`**: validated `train.py` in `LOCAL_CONTAINER` mode, then ran it at scale as a managed training job
- **Explainability**: selected the best candidate with `mlflow.search_logged_models`, then ran [`pipeline_steps/explain.py`](pipeline_steps/explain.py) as a managed job via `@remote` to produce SHAP attributions and standardized fairness metrics, logged onto the model's own MLflow run
- **Registration**: resolved the job's MLflow logged model, logged an `inference.py` and an **inference specification** into its artifact store, then called `mlflow.register_model()` — so the auto-synced SageMaker Model Package arrived **already deployable**, serving directly from the MLflow artifact store
- **Deployment**: validated the logged `inference.py` by importing it in this kernel exactly as the serving container does, approved the Model Package version, and deployed **that version** to a real-time endpoint with `Model` / `EndpointConfig` / `Endpoint`, then scored the test set

Each capability was exercised once, there is exactly one registration, and the endpoint serves the registered and approved package rather than a separately built artifact.

In the next notebook you combine these processing, training, and registration steps into an **automated ML pipeline** with SageMaker Pipelines — where [`pipeline_steps/register.py`](pipeline_steps/register.py) performs the same inference-specification logging you just did by hand, and a `ConditionStep` decides whether the model is registered at all. The explainability function you just ran with `@remote` slots straight in as a `@step` — same code, different decorator.

## Continue with step 3
Open the step 3 [notebook](03-sagemaker-pipeline.ipynb).

## Further development ideas for your real-world projects
- Learn more about [managing experiments with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- Try no-code [SageMaker Canvas](https://docs.aws.amazon.com/sagemaker/latest/dg/canvas.html) on your data to perform analysis and use automated ML to build models and generate predictions.
- To improve the model training, try [automated hyperparameter tuning](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html).

## Additional resources
- [Managing ML experiments with MLflow in SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- [MLflow Apps in SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup.html)
- [Using Docker containers with SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html)
- [SageMaker JupyterLab guide](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-jl.html)
- [Operationalize your Amazon SageMaker Studio notebooks as scheduled notebook jobs](https://aws.amazon.com/blogs/machine-learning/operationalize-your-amazon-sagemaker-studio-notebooks-as-scheduled-notebook-jobs/)
- [scikit-learn dataset transformations](https://scikit-learn.org/stable/data_transforms.html)
- [Amazon SageMaker Immersion Day](https://catalog.us-east-1.prod.workshops.aws/workshops/63069e26-921c-4ce1-9cc7-dd882ff62575/en-US)

### Hands-on examples
- [Official Amazon SageMaker Examples repository](https://github.com/aws/amazon-sagemaker-examples)
- [Community Amazon SageMaker Examples repository](https://github.com/aws/amazon-sagemaker-examples-community)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>